[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/05_machine_learning_fundamentals/03_train_validation_test_and_model_selection.ipynb)

# Bài 03 — Train / Validation / Test và lựa chọn model

> Tương ứng các bài *"Đánh giá hàm giả thuyết & Lựa chọn mô hình phù hợp"* và *"Train / Validation / Test Sets"* trong kế hoạch Tuần 6 của khoá AIVN.

**🎯 Mục tiêu**
- Hiểu vì sao lỗi trên tập train **không** cho biết model tốt đến đâu.
- Vai trò khác nhau của train, validation, test; tỉ lệ chia hợp lý.
- Chọn model / hyperparameter bằng validation; vì sao **chọn trên test làm điểm số lạc quan giả tạo**.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/05_machine_learning_fundamentals

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

from ml_course.visualization import set_style

set_style()
rng = np.random.default_rng(1)

## 1. Lỗi train là thước đo lạc quan

Mục tiêu thật của ML là **generalization** — dự đoán tốt trên dữ liệu **chưa từng thấy**. Model có thể "học thuộc" dữ liệu train: lỗi train rất nhỏ nhưng dự đoán dữ liệu mới rất tệ.

Ví dụ: dữ liệu sinh từ hàm $f(x) = \sin(1.5x)$ cộng nhiễu. Fit đa thức bậc 1, 4, 15:

In [ ]:
def true_f(x):
    return np.sin(1.5 * x)


n = 30
x = np.sort(rng.uniform(-3, 3, n))
y = true_f(x) + rng.normal(0, 0.3, n)
x_new = rng.uniform(-3, 3, 1000)                 # "dữ liệu tương lai"
y_new = true_f(x_new) + rng.normal(0, 0.3, 1000)


def poly_model(degree):
    return make_pipeline(PolynomialFeatures(degree), StandardScaler(), LinearRegression())


grid = np.linspace(-3, 3, 300)
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
for ax, d in zip(axes, [1, 4, 15]):
    m = poly_model(d).fit(x[:, None], y)
    train_mse = mean_squared_error(y, m.predict(x[:, None]))
    new_mse = mean_squared_error(y_new, m.predict(x_new[:, None]))
    ax.scatter(x, y, s=20, label="train")
    ax.plot(grid, true_f(grid), "k--", lw=1, label="hàm thật")
    ax.plot(grid, m.predict(grid[:, None]), "r", lw=2, label=f"bậc {d}")
    ax.set_ylim(-2.5, 2.5)
    ax.set_title(f"bậc {d}: MSE train = {train_mse:.3f} | MSE dữ liệu mới = {new_mse:.3f}")
    ax.legend(fontsize=8)
plt.show()

Bậc 15 có lỗi train **nhỏ nhất** nhưng lỗi trên dữ liệu mới **lớn nhất** — overfitting (bài 04). Vậy ta cần một cách **ước lượng lỗi trên dữ liệu mới** khi chưa có dữ liệu mới → **giữ lại một phần dữ liệu** mà model không được học.

## 2. Ba tập dữ liệu, ba vai trò

| Tập | Dùng để | Được "nhìn" bao nhiêu lần? |
|---|---|---|
| **Train** | học tham số (`fit`) | liên tục |
| **Validation** (dev) | chọn model, chọn hyperparameter, early stopping, chọn feature | nhiều lần → dần bị "overfit" theo |
| **Test** | ước lượng **cuối cùng**, không thiên lệch, hiệu năng trên dữ liệu mới | **một lần**, ở cuối |

```text
Toàn bộ dữ liệu
├── Train (60–80%) ──► fit
├── Validation (10–20%) ──► so sánh & chọn
└── Test (10–20%) ──► báo cáo một lần cuối cùng   ⛔ KHÔNG dùng để tune
```

Khi dữ liệu ít, thay validation cố định bằng **cross-validation** trên phần train (chương 15).

**Tỉ lệ chia:** dữ liệu vài nghìn mẫu → 60/20/20 hoặc 70/15/15. Hàng triệu mẫu → 98/1/1 vẫn đủ (1% của 10 triệu = 100 000 mẫu là quá đủ để đánh giá chính xác).

In [ ]:
X_all = rng.uniform(-3, 3, 300)[:, None]
y_all = true_f(X_all[:, 0]) + rng.normal(0, 0.3, 300)

X_temp, X_test, y_temp, y_test = train_test_split(X_all, y_all, test_size=0.2, random_state=0)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=0)   # 0.25 × 0.8 = 0.2
print(f"train: {len(X_train)} | validation: {len(X_val)} | test: {len(X_test)}")

## 3. Lựa chọn model bằng validation

Chọn bậc đa thức tốt nhất: fit trên train, đo lỗi trên validation, chọn bậc có lỗi validation nhỏ nhất. Cuối cùng mới đo **một lần** trên test.

In [ ]:
degrees = range(1, 16)
train_err, val_err = [], []
for d in degrees:
    m = poly_model(d).fit(X_train, y_train)
    train_err.append(mean_squared_error(y_train, m.predict(X_train)))
    val_err.append(mean_squared_error(y_val, m.predict(X_val)))

best_d = degrees[int(np.argmin(val_err))]
fig, ax = plt.subplots()
ax.plot(degrees, train_err, "o-", label="train")
ax.plot(degrees, val_err, "s-", label="validation")
ax.axvline(best_d, color="gray", ls="--", label=f"chọn bậc {best_d}")
ax.set_xlabel("bậc đa thức (độ phức tạp của model)")
ax.set_ylabel("MSE")
ax.set_yscale("log")
ax.legend()
ax.set_title("Train error giảm mãi; validation error có dạng chữ U")
plt.show()

final = poly_model(best_d).fit(X_temp, y_temp)       # (tuỳ chọn) fit lại trên train + validation
print(f"Bậc được chọn: {best_d}")
print(f"MSE test (báo cáo một lần): {mean_squared_error(y_test, final.predict(X_test)):.4f}  — nhiễu thật có MSE ≈ {0.3**2:.2f}")

Sau khi chọn xong hyperparameter, thường **fit lại** model trên train + validation (thêm dữ liệu) trước khi đánh giá trên test và triển khai.

## 4. Vì sao không được chọn model trên test?

Nếu thử 100 cấu hình và chọn cấu hình có điểm test cao nhất, điểm đó **lạc quan giả tạo**: một phần cao nhờ **may mắn** với chính tập test này. Mô phỏng cực đoan: 100 "model" **đoán ngẫu nhiên** (accuracy thật = 50%) trên test 200 mẫu:

In [ ]:
y_test_cls = rng.integers(0, 2, 200)
random_accs = [np.mean(rng.integers(0, 2, 200) == y_test_cls) for _ in range(100)]
print("accuracy thật của mọi model: 0.50")
print(f"accuracy TỐT NHẤT trên test sau khi chọn 1/100 model: {max(random_accs):.3f}  ← ảo tưởng!")

fresh = np.mean(rng.integers(0, 2, 200) == rng.integers(0, 2, 200))
print(f"model 'tốt nhất' đó trên dữ liệu mới: ~{fresh:.3f}")

Đây là **overfitting vào tập test** (còn gọi là *test set reuse*, hay "Kaggle leaderboard overfitting"). Cùng hiện tượng xảy ra với validation — nên khi tune quá nhiều, điểm validation cũng lạc quan; test giữ riêng là chốt chặn cuối cùng.

## 5. Những lưu ý khi chia dữ liệu

- **Classification mất cân bằng** → chia **stratified** (giữ tỉ lệ lớp): `train_test_split(..., stratify=y)`.
- **Dữ liệu theo thời gian** → train là quá khứ, test là tương lai — **không xáo trộn ngẫu nhiên** (chương 25).
- **Dữ liệu theo nhóm** (nhiều ảnh của cùng một bệnh nhân, nhiều giao dịch của cùng một khách hàng) → chia theo **nhóm**, không để một nhóm nằm ở cả train và test (chương 15, bài 07 về leakage).
- Validation/test phải **giống phân phối dữ liệu khi triển khai** (bài 08).
- Cố định `random_state` để tái lập được.

## 🧠 Kiểm tra nhanh

1. Model A: MSE train 0.01, validation 0.9. Model B: train 0.2, validation 0.25. Chọn model nào? Vì sao?
2. Vì sao test set chỉ nên dùng một lần?
3. Có 5 triệu mẫu. Tỉ lệ chia 98/1/1 có hợp lý không?
4. Dữ liệu doanh số theo ngày: vì sao không nên chia train/test ngẫu nhiên?

## 📌 Tóm tắt

- Lỗi train đánh giá thấp lỗi thật; cần dữ liệu "giữ lại" để ước lượng generalization.
- Train → học; validation → chọn; test → báo cáo một lần.
- Chọn trên test (hoặc tune quá nhiều trên validation) → điểm số lạc quan giả tạo.
- Chia stratified / theo thời gian / theo nhóm tuỳ cấu trúc dữ liệu.

## 📚 Tài liệu tham khảo

- Andrew Ng — *Machine Learning Yearning*, chương 5–12 (dev/test sets)
- Hastie, Tibshirani, Friedman — *The Elements of Statistical Learning*, §7.2